### Data Ingestion

In [1]:
### Document Structure

from langchain_core.documents import Document

In [2]:
doc=Document(
    page_content="This is the main text content I am using to create a RAG",
    metadata={
        "source":"example.txt",
        "page":1,
        "author":"Sanjiv Paul",
        "date_created":"2026-01-24"
    }
)

doc

Document(metadata={'source': 'example.txt', 'page': 1, 'author': 'Sanjiv Paul', 'date_created': '2026-01-24'}, page_content='This is the main text content I am using to create a RAG')

In [3]:
### Creating a simple text file

import os
os.makedirs("../data/text_files", exist_ok=True)

In [4]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction
    
    Python is a high-level, interpreted programming language known for its simplicity.
    Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
    programming languages in the world.

    Key Features:
    - Easy to learn and use
    - Extensive standard library
    - Cross-platform compatibility
    - Strong community support

    Python is widely used in web development, data analysis, artificial intelligence, scientific computing, automation and more.
    """,
    "../data/text_files/machine_learning.txt":"""Machine Learning Basics
    
    Machine Learning (ML) is a subset of artificial intelligence that focuses on building systems that can learn from and make decisions based on data.
    Instead of being explicitly programmed for every task, ML algorithms identify patterns in data and improve their performance over time.

    Types of Machine Learning:
    1. Supervised Learning: Learning with labeled data.
    2. Unsupervised Learning: Finding patterns in unlabled data.
    3. Reinforcement Learning: Learning through rewards and penalties.

    Applications includes image recognition, natural language processing, recommendation systems, speech processing and more.
    """
}

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created!")

Sample text files created!


In [5]:
### TextLoader

# from langchain.document_loaders import TextLoad

from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
document=loader.load()
print(document)

/Users/sanjiv/Developer/Projects/project_45_RAG/RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\n    Python is a high-level, interpreted programming language known for its simplicity.\n    Created by Guido van Rossum and first released in 1991, Python has become one of the most popular\n    programming languages in the world.\n\n    Key Features:\n    - Easy to learn and use\n    - Extensive standard library\n    - Cross-platform compatibility\n    - Strong community support\n\n    Python is widely used in web development, data analysis, artificial intelligence, scientific computing, automation and more.\n    ')]


In [6]:
### Directory Loader

from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the dir
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", # pattern to match files
    loader_cls=TextLoader, # loader class to use
    loader_kwargs={"encoding":"utf-8"}, # kwargs for the loader class
    show_progress=False
)

documents=dir_loader.load()
print(f"Total documents loaded: {len(documents)}")
# for doc in documents:
#     print(doc)

print(documents)

Total documents loaded: 2
[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\n    Python is a high-level, interpreted programming language known for its simplicity.\n    Created by Guido van Rossum and first released in 1991, Python has become one of the most popular\n    programming languages in the world.\n\n    Key Features:\n    - Easy to learn and use\n    - Extensive standard library\n    - Cross-platform compatibility\n    - Strong community support\n\n    Python is widely used in web development, data analysis, artificial intelligence, scientific computing, automation and more.\n    '), Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\n    Machine Learning (ML) is a subset of artificial intelligence that focuses on building systems that can learn from and make decisions based on data.\n    Instead of being explicitly programmed for every task, ML al

In [7]:
### PDF Loaders

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the dir
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", # pattern to match files
    loader_cls=PyMuPDFLoader, # loader class to use
    show_progress=False
)

pdf_documents = dir_loader.load()
pdf_documents


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-12-30T09:23:59+00:00', 'source': '../data/pdf/objectdetection.pdf', 'file_path': '../data/pdf/objectdetection.pdf', 'total_pages': 60, 'format': 'PDF 1.5', 'title': 'Microsoft Word - final', 'author': 'K.SAI NAVEEN', 'subject': '', 'keywords': '', 'moddate': '2020-12-30T09:23:59+00:00', 'trapped': '', 'modDate': 'D:20201230092359Z', 'creationDate': "D:20201230092359+00'00'", 'page': 0}, page_content='1 \n \n \n \nREAL TIME OBJECT DETECTION USING DEEP LEARNING \n \nA Project report submitted in partial fulfillment of the requirements for the award of the \ndegree of \n \nBACHELOR OF TECHNOLOGY \n \n \nIN \n \nELECTRONICS AND COMMUNICATION ENGINEERING \n \n \nSubmitted by \n \nD Pavan (316126512073) \nV S Ashlesh Kumar (31612651119) \nJ.A.S. Sampreeth (316126512139) \nK. Sai Naveen (316126512141) \n \nUnder the guidance of \n \nMs. Ch. Padma Sree, M.Tech, (Ph.D) \n \nAssistant Pro

### embedding And vectorStoreDB

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts:List[str]) -> np.ndarray:
        """
        Generating embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_din)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    # we dont need function yet
    def get_embedding_dimentions(self)->int:
        """Get the embedding dimentions of the model"""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()
    

## initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1569.47it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [ ]:
class VectorStore:
    """Manages document embeddings in ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory:str = "../data/vector_store"):
        """ 
        Intialize the vector store

        Args:
            collection_name: Name of the chromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF document for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    
    def add_documents(self, documents: List[Any], embeddings:np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")


        # Prepare data for chromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total docuemnts in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
